# MTSamples HTML Parser

Parses the per-record HTML files from the Kaggle `mtsamples-dataset` (shoubhikchakra/mtsamples-dataset) into clean structured rows, and extracts medication/plan-relevant sentences for NER annotation.

**Input:** a folder of `N.html` files, e.g. `mtsamples/content/mtsamples/0.html`, `1.html`, ...

**Output:**
- `parsed_mtsamples.csv` — one row per report (specialty, sample name, description, all sections)
- `ner_candidate_sentences.csv` — sentence-level rows from medication/plan sections, ready for Label Studio


## 1. Install & import dependencies

In [ ]:
!pip install beautifulsoup4 -q

In [ ]:
import csv
import re
from pathlib import Path
from bs4 import BeautifulSoup

## 2. Set your paths

Edit `INPUT_DIR` below to point at your local folder of HTML files.

In [ ]:
INPUT_DIR = r"C:\Users\rohit\Downloads\mtsamples\content\mtsamples"
OUTPUT_CSV = "parsed_mtsamples.csv"
SENTENCES_CSV = "ner_candidate_sentences.csv"
LIMIT = 200  # set to None to process all files once you've sanity-checked the output

## 3. Define the high-value sections and helpers

In [ ]:
# Sections that are high-value for medication/dosage/instruction NER.
# We keep these; everything else (SUBJECTIVE/OBJECTIVE/HISTORY/etc.) is
# still saved in the full record but excluded from the NER candidate pool
# since it's mostly narrative prose with low drug-entity density.
HIGH_VALUE_SECTIONS = {
    "MEDICATIONS",
    "DISCHARGE MEDICATIONS",
    "CURRENT MEDICATIONS",
    "PLAN",
    "TREATMENT",
    "RECOMMENDATIONS",
    "PRESCRIPTIONS",
    "ASSESSMENT AND PLAN",
    "DISPOSITION",
}

SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")

In [ ]:
def parse_record(html_path: Path) -> dict:
    """Extract specialty, sample name, description, and section->text map."""
    raw = html_path.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(raw, "html.parser")
    text = soup.get_text("\n")

    # Normalize whitespace / stray unicode artifacts (Â, non-breaking spaces)
    text = text.replace("\xa0", " ").replace("Â", "")
    lines = [l.strip() for l in text.split("\n") if l.strip()]

    record = {
        "file": html_path.name,
        "specialty": "",
        "sample_name": "",
        "description": "",
        "sections": {},
    }

    # Pull specialty / sample name / description via labeled prefixes
    full_text = "\n".join(lines)
    m = re.search(r"Medical Specialty:\*\n?(.+)", full_text)
    if m:
        record["specialty"] = m.group(1).strip()
    m = re.search(r"Sample Name:\*(.+)", full_text)
    if m:
        record["sample_name"] = m.group(1).strip()
    m = re.search(r"Description:\*(.+)", full_text)
    if m:
        record["description"] = m.group(1).strip()

    # Body is split into SECTION: text blocks. Detect headers as lines
    # that are ALL CAPS (with optional trailing colon), e.g. "MEDICATIONS:"
    header_re = re.compile(r"^([A-Z][A-Z /]+):\s*(.*)$")
    current_section = None
    buffer = []
    for line in lines:
        header_match = header_re.match(line)
        if header_match and header_match.group(1) in _known_headers(full_text):
            if current_section:
                record["sections"][current_section] = " ".join(buffer).strip()
            current_section = header_match.group(1)
            buffer = [header_match.group(2)] if header_match.group(2) else []
        else:
            if current_section:
                buffer.append(line)
    if current_section:
        record["sections"][current_section] = " ".join(buffer).strip()

    return record


def _known_headers(full_text: str):
    """Heuristic: treat any short ALL-CAPS line ending in ':' as a header."""
    return set(re.findall(r"^([A-Z][A-Z /]{2,40}):", full_text, flags=re.MULTILINE))


def extract_ner_candidate_sentences(record: dict) -> list:
    """Pull sentences from high-value sections for annotation."""
    candidates = []
    for section, text in record["sections"].items():
        if section in HIGH_VALUE_SECTIONS and text:
            for sent in SENTENCE_SPLIT_RE.split(text):
                sent = sent.strip()
                if len(sent.split()) >= 4:  # skip fragments
                    candidates.append({
                        "file": record["file"],
                        "specialty": record["specialty"],
                        "section": section,
                        "sentence": sent,
                    })
    return candidates

## 4. Run the parser over all HTML files

In [ ]:
input_dir = Path(INPUT_DIR)
html_files = sorted(input_dir.glob("*.html"), key=lambda p: int(p.stem) if p.stem.isdigit() else 0)
if LIMIT:
    html_files = html_files[:LIMIT]

print(f"Found {len(html_files)} HTML files to process.")

In [ ]:
all_records = []
all_sentences = []

for hp in html_files:
    try:
        rec = parse_record(hp)
    except Exception as e:
        print(f"  ! skipped {hp.name}: {e}")
        continue
    all_records.append(rec)
    all_sentences.extend(extract_ner_candidate_sentences(rec))

print(f"Parsed {len(all_records)} records.")
print(f"Extracted {len(all_sentences)} NER-candidate sentences from high-value sections.")

## 5. Write output CSVs

In [ ]:
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["file", "specialty", "sample_name", "description", "sections"])
    for rec in all_records:
        writer.writerow([
            rec["file"], rec["specialty"], rec["sample_name"], rec["description"],
            " | ".join(f"{k}::{v}" for k, v in rec["sections"].items()),
        ])

with open(SENTENCES_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["file", "specialty", "section", "sentence"])
    for row in all_sentences:
        writer.writerow([row["file"], row["specialty"], row["section"], row["sentence"]])

print(f"-> {OUTPUT_CSV}")
print(f"-> {SENTENCES_CSV}")

## 6. Quick sanity check

In [ ]:
import pandas as pd

df_sentences = pd.read_csv(SENTENCES_CSV)
print(df_sentences.shape)
df_sentences.head(15)

In [ ]:
df_sentences["section"].value_counts()